<div style="
  background: linear-gradient(145deg, #0f172a, #1e293b);
  border: 4px solid transparent;
  border-radius: 14px;
  padding: 18px 22px;
  margin: 12px 0;
  font-size: 36px;
  font-weight: 600;
  color: #f8fafc;
  box-shadow: 0 6px 14px rgba(0,0,0,0.25);
  background-clip: padding-box;
  position: relative;
">
  <div style="
    position: absolute;
    inset: 0;
    padding: 4px;
    border-radius: 14px;
    background: linear-gradient(90deg, #06b6d4, #3b82f6, #8b5cf6);
    -webkit-mask: 
      linear-gradient(#fff 0 0) content-box, 
      linear-gradient(#fff 0 0);
    -webkit-mask-composite: xor;
    mask-composite: exclude;
    pointer-events: none;
  "></div>
  <b>03. Hierarchical Bayesian Models</b>
</div>

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">📑 Table of Contents</span>

- **1. [Why Hierarchical Models?](#-part-i-why-hierarchical-models)**
    - 1.1 [Pooling Strategies](#pooling-strategies)
    - 1.2 [The Partial Pooling Advantage](#partial-pooling-advantage)
- **2. [Setup & Imports](#-part-ii-setup--imports)**
- **3. [Building Hierarchical Models with TFP](#-part-iii-building-hierarchical-models)**
    - 3.1 [JointDistribution API](#jointdistribution-api)
    - 3.2 [Hierarchical Linear Regression](#hierarchical-linear-regression)
- **4. [Inference with MCMC](#-part-iv-inference-with-mcmc)**
- **5. [Multi-Level Model: Schools Example](#-part-v-schools-example)**
- **6. [Key Takeaways](#-part-vi-key-takeaways)**

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">1. Why Hierarchical Models?</span>

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #312e81, #111827); padding: 10px 18px; border-radius: 10px; font-size: 24px; font-weight: 600; box-shadow: 0 3px 10px rgba(0,0,0,0.15);">1.1. Pooling Strategies</span>

When data comes from multiple groups (e.g., students from different schools, patients from different hospitals):

| **Strategy** | **Description** | **Problem** |
|-------------|----------------|-------------|
| **Complete Pooling** | Ignore group structure, fit one model | Ignores group-level variation |
| **No Pooling** | Fit separate model per group | Overfits in small groups |
| **Partial Pooling** ✅ | Share information across groups | **Best of both worlds** |

Hierarchical (multi-level) models implement **partial pooling** by placing priors on group-level parameters.

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #312e81, #111827); padding: 10px 18px; border-radius: 10px; font-size: 24px; font-weight: 600; box-shadow: 0 3px 10px rgba(0,0,0,0.15);">1.2. The Partial Pooling Advantage</span>

The hierarchical structure:

$$\mu_j \sim \text{Normal}(\mu_0, \tau) \quad \text{(group-level)}$$
$$y_{ij} \sim \text{Normal}(\mu_j, \sigma) \quad \text{(observation-level)}$$

Where:
- $\mu_0, \tau$ are **hyperparameters** (learned from data)
- $\mu_j$ are **group-level parameters** (partially pooled toward $\mu_0$)
- Groups with less data are pulled more toward the global mean (**shrinkage**)

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">2. Setup & Imports</span>

In [ ]:
import tensorflow as tf
import tensorflow_probability as tfp
import numpy as np
import matplotlib.pyplot as plt

tfd = tfp.distributions
tfb = tfp.bijectors

tf.random.set_seed(42)
np.random.seed(42)

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

print(f"TensorFlow: {tf.__version__}")
print(f"TFP: {tfp.__version__}")

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">3. Building Hierarchical Models with TFP</span>

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #312e81, #111827); padding: 10px 18px; border-radius: 10px; font-size: 24px; font-weight: 600; box-shadow: 0 3px 10px rgba(0,0,0,0.15);">3.1. JointDistribution API</span>

TFP provides three ways to build joint distributions:

| **API** | **Style** | **Best For** |
|--------|----------|-------------|
| `JointDistributionSequential` | List-based | Simple chains |
| `JointDistributionNamed` | Dict-based | Named parameters |
| `JointDistributionCoroutine` | Generator-based | Complex dependencies |

In [ ]:
# Example: Simple hierarchical model with JointDistributionSequential
simple_hierarchical = tfd.JointDistributionSequential([
    # Hyperprior: global mean
    tfd.Normal(loc=0., scale=10.),        # mu_0
    # Hyperprior: between-group std
    tfd.HalfNormal(scale=5.),              # tau
    # Group-level parameters (5 groups)
    lambda tau, mu_0: tfd.Independent(
        tfd.Normal(loc=tf.fill([5], mu_0), scale=tau), 1
    ),
])

# Sample from the prior
prior_samples = simple_hierarchical.sample(3)
print("Prior samples from hierarchical model:")
print(f"  Global mean (mu_0): {prior_samples[0].numpy()}")
print(f"  Between-group std (tau): {prior_samples[1].numpy()}")
print(f"  Group means: {prior_samples[2].numpy()}")

# Log probability
lp = simple_hierarchical.log_prob(prior_samples)
print(f"  Log prob: {lp.numpy()}")

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #312e81, #111827); padding: 10px 18px; border-radius: 10px; font-size: 24px; font-weight: 600; box-shadow: 0 3px 10px rgba(0,0,0,0.15);">3.2. Hierarchical Linear Regression</span>

In [ ]:
# Generate synthetic hierarchical data
num_groups = 6
true_global_slope = 2.0
true_global_intercept = 1.0
true_slope_std = 0.5
true_intercept_std = 1.0

group_data = []
np.random.seed(42)

for g in range(num_groups):
    n_g = np.random.randint(10, 40)  # Different group sizes
    slope_g = np.random.normal(true_global_slope, true_slope_std)
    intercept_g = np.random.normal(true_global_intercept, true_intercept_std)
    x_g = np.random.uniform(-2, 2, n_g).astype(np.float32)
    y_g = (slope_g * x_g + intercept_g + np.random.normal(0, 0.3, n_g)).astype(np.float32)
    group_data.append({'x': x_g, 'y': y_g, 'n': n_g,
                       'true_slope': slope_g, 'true_intercept': intercept_g})

# Visualize
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
colors = ['r', 'b', 'g', 'orange', 'm', 'c']

for i, (ax, gd) in enumerate(zip(axes.flat, group_data)):
    ax.scatter(gd['x'], gd['y'], c=colors[i], s=30, alpha=0.7, edgecolors='white')
    x_line = np.linspace(-2.5, 2.5, 50)
    ax.plot(x_line, gd['true_slope'] * x_line + gd['true_intercept'], 
            color=colors[i], linewidth=2, alpha=0.8)
    ax.set_title(f"Group {i+1} (n={gd['n']})", fontsize=13, fontweight='bold')
    ax.set_xlabel('x', fontsize=11)
    ax.set_ylabel('y', fontsize=11)

plt.suptitle('Hierarchical Data: 6 Groups with Different Slopes & Intercepts',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">4. Inference with MCMC</span>

In [ ]:
# Flatten all data
x_all = np.concatenate([gd['x'] for gd in group_data])
y_all = np.concatenate([gd['y'] for gd in group_data])
group_ids = np.concatenate([np.full(gd['n'], i) for i, gd in enumerate(group_data)]).astype(np.int32)

x_all_tf = tf.constant(x_all)
y_all_tf = tf.constant(y_all)
group_ids_tf = tf.constant(group_ids)

def hierarchical_log_prob(global_slope, global_intercept,
                          slope_std, intercept_std,
                          group_slopes, group_intercepts, sigma):
    """
    Unnormalized log posterior for hierarchical linear regression.
    """
    # Hyperpriors
    lp = tfd.Normal(0., 10.).log_prob(global_slope)
    lp += tfd.Normal(0., 10.).log_prob(global_intercept)
    lp += tfd.HalfNormal(5.).log_prob(slope_std)
    lp += tfd.HalfNormal(5.).log_prob(intercept_std)
    lp += tfd.HalfNormal(5.).log_prob(sigma)
    
    # Group-level priors
    lp += tf.reduce_sum(tfd.Normal(global_slope, slope_std).log_prob(group_slopes))
    lp += tf.reduce_sum(tfd.Normal(global_intercept, intercept_std).log_prob(group_intercepts))
    
    # Likelihood
    slopes_per_obs = tf.gather(group_slopes, group_ids_tf)
    intercepts_per_obs = tf.gather(group_intercepts, group_ids_tf)
    predictions = slopes_per_obs * x_all_tf + intercepts_per_obs
    lp += tf.reduce_sum(tfd.Normal(predictions, sigma).log_prob(y_all_tf))
    
    return lp

# Use constrained bijectors for positive parameters
kernel = tfp.mcmc.TransformedTransitionKernel(
    inner_kernel=tfp.mcmc.NoUTurnSampler(
        target_log_prob_fn=hierarchical_log_prob,
        step_size=0.05
    ),
    bijector=[
        tfb.Identity(),   # global_slope (unconstrained)
        tfb.Identity(),   # global_intercept
        tfb.Softplus(),   # slope_std (positive)
        tfb.Softplus(),   # intercept_std (positive)
        tfb.Identity(),   # group_slopes
        tfb.Identity(),   # group_intercepts
        tfb.Softplus(),   # sigma (positive)
    ]
)

adaptive_kernel = tfp.mcmc.SimpleStepSizeAdaptation(
    inner_kernel=kernel,
    num_adaptation_steps=400,
    target_accept_prob=0.75
)

@tf.function
def run_hierarchical_mcmc():
    initial_state = [
        tf.constant(0.0),              # global_slope
        tf.constant(0.0),              # global_intercept
        tf.constant(1.0),              # slope_std
        tf.constant(1.0),              # intercept_std
        tf.zeros(num_groups),          # group_slopes
        tf.zeros(num_groups),          # group_intercepts
        tf.constant(1.0),              # sigma
    ]
    return tfp.mcmc.sample_chain(
        num_results=2000,
        num_burnin_steps=1000,
        current_state=initial_state,
        kernel=adaptive_kernel,
        trace_fn=None
    )

hier_samples = run_hierarchical_mcmc()

print("Hierarchical model posterior estimates:")
print(f"  Global slope:     {tf.reduce_mean(hier_samples[0]):.3f} ± {tf.math.reduce_std(hier_samples[0]):.3f}  (true: {true_global_slope})")
print(f"  Global intercept: {tf.reduce_mean(hier_samples[1]):.3f} ± {tf.math.reduce_std(hier_samples[1]):.3f}  (true: {true_global_intercept})")
print(f"  Slope std:        {tf.reduce_mean(hier_samples[2]):.3f} ± {tf.math.reduce_std(hier_samples[2]):.3f}  (true: {true_slope_std})")
print(f"  Intercept std:    {tf.reduce_mean(hier_samples[3]):.3f} ± {tf.math.reduce_std(hier_samples[3]):.3f}  (true: {true_intercept_std})")

In [ ]:
# Visualize shrinkage effect
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Slopes
true_slopes = [gd['true_slope'] for gd in group_data]
post_slopes = tf.reduce_mean(hier_samples[4], axis=0).numpy()
no_pool_slopes = [np.polyfit(gd['x'], gd['y'], 1)[0] for gd in group_data]

x_pos = np.arange(num_groups)
width = 0.25

axes[0].bar(x_pos - width, true_slopes, width, label='True', color='r', alpha=0.8)
axes[0].bar(x_pos, no_pool_slopes, width, label='No Pooling (OLS)', color='gray', alpha=0.8)
axes[0].bar(x_pos + width, post_slopes, width, label='Partial Pooling (Hierarchical)', color='b', alpha=0.8)
axes[0].axhline(y=true_global_slope, color='red', linestyle='--', alpha=0.5, label='Global mean')
axes[0].set_xlabel('Group', fontsize=13)
axes[0].set_ylabel('Slope', fontsize=13)
axes[0].set_title('Shrinkage Effect: Slopes', fontsize=15, fontweight='bold')
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels([f'G{i+1}\n(n={gd["n"]})' for i, gd in enumerate(group_data)])
axes[0].legend(fontsize=10)

# Intercepts
true_intercepts = [gd['true_intercept'] for gd in group_data]
post_intercepts = tf.reduce_mean(hier_samples[5], axis=0).numpy()
no_pool_intercepts = [np.polyfit(gd['x'], gd['y'], 1)[1] for gd in group_data]

axes[1].bar(x_pos - width, true_intercepts, width, label='True', color='r', alpha=0.8)
axes[1].bar(x_pos, no_pool_intercepts, width, label='No Pooling (OLS)', color='gray', alpha=0.8)
axes[1].bar(x_pos + width, post_intercepts, width, label='Partial Pooling (Hierarchical)', color='g', alpha=0.8)
axes[1].axhline(y=true_global_intercept, color='red', linestyle='--', alpha=0.5, label='Global mean')
axes[1].set_xlabel('Group', fontsize=13)
axes[1].set_ylabel('Intercept', fontsize=13)
axes[1].set_title('Shrinkage Effect: Intercepts', fontsize=15, fontweight='bold')
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels([f'G{i+1}\n(n={gd["n"]})' for i, gd in enumerate(group_data)])
axes[1].legend(fontsize=10)

plt.suptitle('Partial Pooling Shrinks Estimates Toward the Global Mean\n(especially for smaller groups)', 
             fontsize=16, fontweight='bold', y=1.04)
plt.tight_layout()
plt.show()

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">5. Multi-Level Model: Eight Schools Example</span>

The classic "Eight Schools" problem from Rubin (1981) — estimating the effect of coaching programs across 8 schools.

In [ ]:
# Eight Schools data (Rubin, 1981)
treatment_effects = tf.constant([28., 8., -3., 7., -1., 1., 18., 12.])
treatment_stddevs = tf.constant([15., 10., 16., 11., 9., 11., 10., 18.])
num_schools = 8

# Non-centered parameterization (better for HMC)
def eight_schools_log_prob(mu, log_tau, theta_raw):
    """
    Non-centered parameterization:
    theta_j = mu + tau * theta_raw_j
    """
    tau = tf.exp(log_tau)
    theta = mu + tau * theta_raw
    
    # Hyperpriors
    lp = tfd.Normal(0., 50.).log_prob(mu)
    lp += tfd.HalfCauchy(0., 10.).log_prob(tau) + log_tau  # Jacobian
    
    # Group parameters (standard normal in non-centered form)
    lp += tf.reduce_sum(tfd.Normal(0., 1.).log_prob(theta_raw))
    
    # Likelihood
    lp += tf.reduce_sum(tfd.Normal(theta, treatment_stddevs).log_prob(treatment_effects))
    
    return lp

# MCMC sampling
schools_kernel = tfp.mcmc.SimpleStepSizeAdaptation(
    inner_kernel=tfp.mcmc.NoUTurnSampler(
        target_log_prob_fn=eight_schools_log_prob,
        step_size=0.1
    ),
    num_adaptation_steps=600,
    target_accept_prob=0.80
)

@tf.function
def run_eight_schools():
    return tfp.mcmc.sample_chain(
        num_results=4000,
        num_burnin_steps=1000,
        current_state=[
            tf.constant(0.),
            tf.constant(0.),
            tf.zeros(num_schools)
        ],
        kernel=schools_kernel,
        trace_fn=None
    )

mu_samples, log_tau_samples, theta_raw_samples = run_eight_schools()
tau_samples = tf.exp(log_tau_samples)
theta_samples = mu_samples[:, tf.newaxis] + tau_samples[:, tf.newaxis] * theta_raw_samples

print(f"Global mean (mu): {tf.reduce_mean(mu_samples):.2f} ± {tf.math.reduce_std(mu_samples):.2f}")
print(f"Between-school std (tau): {tf.reduce_mean(tau_samples):.2f} ± {tf.math.reduce_std(tau_samples):.2f}")
print(f"\nSchool-level effects (posterior mean ± std):")
for j in range(num_schools):
    mean_j = tf.reduce_mean(theta_samples[:, j])
    std_j = tf.math.reduce_std(theta_samples[:, j])
    print(f"  School {j+1}: {mean_j:.2f} ± {std_j:.2f}  (observed: {treatment_effects[j]:.0f} ± {treatment_stddevs[j]:.0f})")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Forest plot
school_names = [f'School {i+1}' for i in range(num_schools)]
post_means = tf.reduce_mean(theta_samples, axis=0).numpy()
post_stds = tf.math.reduce_std(theta_samples, axis=0).numpy()

y_pos = np.arange(num_schools)

axes[0].errorbar(treatment_effects.numpy(), y_pos + 0.15, 
                 xerr=treatment_stddevs.numpy(), fmt='o', color='r',
                 markersize=8, capsize=5, label='Observed')
axes[0].errorbar(post_means, y_pos - 0.15, 
                 xerr=1.96*post_stds, fmt='s', color='b',
                 markersize=8, capsize=5, label='Hierarchical posterior')
axes[0].axvline(x=tf.reduce_mean(mu_samples).numpy(), color='g', 
                linestyle='--', linewidth=2, label='Global mean')
axes[0].set_yticks(y_pos)
axes[0].set_yticklabels(school_names)
axes[0].set_xlabel('Treatment Effect', fontsize=13)
axes[0].set_title('Eight Schools: Shrinkage Toward Global Mean', fontsize=15, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].invert_yaxis()

# Tau posterior
axes[1].hist(tau_samples.numpy(), bins=50, density=True, alpha=0.7, 
             color='m', edgecolor='white')
axes[1].set_xlabel('τ (between-school std dev)', fontsize=13)
axes[1].set_ylabel('Density', fontsize=13)
axes[1].set_title('Posterior of Between-School Variation (τ)', fontsize=15, fontweight='bold')

plt.tight_layout()
plt.show()

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">6. Key Takeaways</span>

| **Concept** | **Key Point** |
|------------|---------------|
| Partial Pooling | Shares information across groups, avoiding overfitting |
| Shrinkage | Groups with less data are pulled toward the global mean |
| Non-centered parameterization | Essential for efficient HMC sampling |
| JointDistribution API | TFP's tool for building complex probabilistic models |
| Hyperparameters | Learned from data, not fixed — the Bayesian way |

### 🎯 When to Use Hierarchical Models
- Data has **natural grouping structure** (students/schools, patients/hospitals)
- Some groups have **limited data** → shrinkage helps
- You want to **borrow strength** across groups
- You need **group-level and population-level** inference simultaneously